### 1 .Load Data

In [38]:
import pandas as pd
import joblib
import numpy as np
import pandas as pd
import warnings
import joblib
import re
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')

df = pd.read_csv('laptops_train.csv')

### 2. Bersihkan Kolom

In [39]:
df.columns = df.columns.str.strip()

In [40]:
df['Screen_Size'] = df['Screen Size'].str.extract(r'(\d+\.\d+)').astype(float)
df['RAM_GB'] = df['RAM'].str.extract(r'(\d+)').astype(float)
df['Weight_KG'] = df['Weight'].str.extract(r'(\d+\.\d+)').astype(float)
df['Storage_GB'] = df['Storage'].str.extract(r'(\d+)').astype(float)
df['Category'] = df['Category']
df['OS'] = df['Operating System']

In [41]:
# Ambil tier CPU & GPU sederhana
df['CPU_Tier'] = df['CPU'].str.extract(r'(i3|i5|i7|i9|Ryzen|Celeron|Pentium|Atom)')[0].fillna('Other')
df['GPU_Type'] = df['GPU'].str.contains('Intel', case=False, na=False).map({True: 'Intel', False: 'Dedicated'})
df.loc[df['GPU'].str.contains('Nvidia', na=False), 'GPU_Type'] = 'Nvidia'
df.loc[df['GPU'].str.contains('AMD', na=False), 'GPU_Type'] = 'AMD'

### 3. Pilih Fitur & Target

In [42]:
features = ['Category', 'OS', 'CPU_Tier', 'GPU_Type', 'RAM_GB', 'Screen_Size', 'Storage_GB', 'Weight_KG']
x = df[features].copy()
y = df['Price']

In [43]:
# Isi missing value
num_cols = ['RAM_GB', 'Screen_Size', 'Storage_GB', 'Weight_KG']
cat_cols = ['Category', 'OS', 'CPU_Tier', 'GPU_Type']

for col in num_cols:
    x[col] = x[col].fillna(x[col].median()) # Isi lubang angka dengan nilai tengah
for col in cat_cols:
    x[col] = x[col].fillna('Unknown').astype(str) # Isi lubang teks, paksa jadi string
for col in x.select_dtypes(include='object').columns:
    x[col] = x[col].astype(str)

### 4. Encode Kategori

In [44]:
cat_cols = x.select_dtypes(include='object').columns
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
x_encoded = pd.DataFrame(encoder.fit_transform(x[cat_cols]), columns=encoder.get_feature_names_out(cat_cols))
x_final = pd.concat([x.drop(columns=cat_cols), x_encoded], axis=1)

### 5. Train/Test Model

In [53]:
x_train, x_test, y_train, y_test = train_test_split(x_final, y, test_size=0.2, random_state=42)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 15, 20, 25, 30, None],
    'min_samples_split': [2, 5, 8, 10, 15],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.7, 0.8, 1.0],
    'bootstrap': [True, False]
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)
search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,
    cv=3,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1
)

search.fit(x_train, y_train)

# Take the best model
model = search.best_estimator_
print(f"\nBest Parameter: {search.best_params_}")

# Evaluation
y_pred = model.predict(x_test)
train_score = model.score(x_train, y_train)
test_score = model.score(x_test, y_test)




Best Parameter: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 15, 'bootstrap': False}


In [54]:
print("\nEval Result:")
print(f"R² Score   : {r2_score(y_test, y_pred):.4f}")
print(f"MAE (Rp)   : {mean_absolute_error(y_test, y_pred):,.0f}")
print(f"RMSE (Rp)  : {np.sqrt(mean_squared_error(y_test, y_pred)):,.0f}")
print(f"Train Score: {train_score:.4f}")
print(f"Test Score : {test_score:.4f}")


Eval Result:
R² Score   : 0.7382
MAE (Rp)   : 1,835,324
RMSE (Rp)  : 3,076,401
Train Score: 0.9563
Test Score : 0.7382


### 6. Save Model & Encoder

In [55]:
joblib.dump(model, 'model.pkl')
joblib.dump(encoder, 'encoder.pkl')

['encoder.pkl']

### 7. Dropdown List

In [56]:
dropdowns = {}
for col in ['Category', 'OS', 'CPU_Tier', 'GPU_Type']:
    dropdowns[col] = sorted(df[col].dropna().unique().tolist())
for col in ['RAM_GB', 'Screen_Size', 'Storage_GB', 'Weight_KG']:
    dropdowns[col] = sorted(df[col].dropna().unique().tolist())
import json
with open('dropdowns.json', 'w') as f:
    json.dump(dropdowns, f, indent=2)